# Financial VOI Tutorial

A compact worked example showing how voiage can support a finance-flavored
decision problem: compare a few portfolio strategies, inspect downside risk,
and calculate VOI against uncertain market inputs.

In [1]:
import numpy as np

from voiage.analysis import DecisionAnalysis
from voiage.financial.risk_analysis import (
    calculate_sharpe_ratio,
    calculate_value_at_risk,
)
from voiage.schema import ValueArray

rng = np.random.default_rng(42)
strategy_names = ["Conservative", "Balanced", "Growth"]

returns = rng.normal(loc=0.06, scale=0.12, size=500)
var95 = calculate_value_at_risk(returns, confidence_level=0.95)
sharpe = calculate_sharpe_ratio(returns, risk_free_rate=0.02)

print(f"95% VaR on a synthetic return series: {var95:.4f}")
print(f"Sharpe ratio on the same series: {sharpe:.4f}")

95% VaR on a synthetic return series: 0.1298
Sharpe ratio on the same series: 0.3339


In [2]:
rng = np.random.default_rng(42)
n_samples = 500
market_growth = rng.normal(0.05, 0.02, size=n_samples)
volatility = rng.uniform(0.08, 0.18, size=n_samples)
liquidity_pressure = rng.uniform(0.00, 0.20, size=n_samples)

net_benefits = np.column_stack([
    6.0 + 2.5 * market_growth - 4.5 * volatility,
    7.2 + 4.5 * market_growth - 3.5 * volatility - 1.0 * liquidity_pressure,
    8.1 + 6.0 * market_growth - 5.5 * volatility - 1.8 * liquidity_pressure,
])

analysis = DecisionAnalysis(
    nb_array=ValueArray.from_numpy(net_benefits, strategy_names),
    parameter_samples={
        "market_growth": market_growth,
        "volatility": volatility,
        "liquidity_pressure": liquidity_pressure,
    },
)

evpi = analysis.evpi()
evppi_growth = analysis.evppi(["market_growth"])
enbs = analysis.enbs(research_cost=1.5)

print(f"EVPI: {evpi:.4f}")
print(f"EVPPI for market growth: {evppi_growth:.4f}")
print(f"ENBS for a 1.5-unit study cost: {enbs:.4f}")

mean_nb = net_benefits.mean(axis=0)
best_index = int(np.argmax(mean_nb))
print(f"Best strategy by expected net benefit: {strategy_names[best_index]}")
print("Expected net benefits:", dict(zip(strategy_names, np.round(mean_nb, 4), strict=True)))

EVPI: 0.0000
EVPPI for market growth: 0.0000
ENBS for a 1.5-unit study cost: 0.0000
Best strategy by expected net benefit: Growth
Expected net benefits: {'Conservative': np.float64(5.5402), 'Balanced': np.float64(6.8685), 'Growth': np.float64(7.5027)}
